ANÁLISIS, AUDITORÍA Y PREPARACIÓN DE LA BASE DE DATOS

Proyecto 2 – ACA 1

Este notebook documenta el proceso de caracterización, auditoría de calidad, análisis descriptivo y preparación de una base de datos de comunicaciones administrativas correspondientes al año 2025. Se desarrollan procedimientos mediante Python para identificar las características de los registros, evaluar la calidad de la información, analizar las variables y aplicar transformaciones ETL, conservando la trazabilidad de los datos originales.

In [ ]:

import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

# Cargar el archivo de datos
uploaded = files.upload()

nombre_archivo = next(iter(uploaded))

# Leer el archivo CSV
df = pd.read_csv(nombre_archivo, encoding="utf-8-sig")

print("Archivo cargado correctamente.")
print(f"Número de registros: {df.shape[0]:,}")
print(f"Número de variables: {df.shape[1]}")
display(df.head())


In [ ]:

# Estructura de las variables
df.info()

# Tipos de datos
print("\nTipos de datos:")
display(df.dtypes.rename("tipo_dato").to_frame())

# Valores faltantes
faltantes = pd.DataFrame({
    "valores_faltantes": df.isna().sum(),
    "porcentaje_faltante": (df.isna().mean() * 100).round(2)
})

faltantes = faltantes.sort_values(
    "porcentaje_faltante", ascending=False
)

print("\nValores faltantes por variable:")
display(faltantes)

# Registros duplicados exactos
print(
    "\nRegistros duplicados exactos:",
    df.duplicated().sum()
)

# Dimensiones de la base
print("\nDimensiones:", df.shape)


In [ ]:

columnas_fecha = [
    "Fecha radicación",
    "Fecha vencimiento",
    "Fecha Radicado salida"
]

for columna in columnas_fecha:
    print(f"\n--- {columna} ---")
    print("Ejemplos de valores:")
    display(df[columna].dropna().astype(str).head(10))

    fechas_convertidas = pd.to_datetime(
        df[columna],
        errors="coerce",
        dayfirst=True
    )

    invalidas = df[columna].notna() & fechas_convertidas.isna()

    print("Valores no vacíos que no se pudieron interpretar:",
          int(invalidas.sum()))


In [ ]:

# Crear una copia de trabajo para el ETL
df_etl = df.copy()

# 1. Convertir fecha de radicación
df_etl["Fecha radicación"] = pd.to_datetime(
    df_etl["Fecha radicación"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# 2. Convertir fecha de vencimiento
df_etl["Fecha vencimiento"] = pd.to_datetime(
    df_etl["Fecha vencimiento"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# 3. Normalizar fecha de radicado de salida
salida = (
    df_etl["Fecha Radicado salida"]
    .astype("string")
    .str.strip()
    .str.replace("\u00a0", " ", regex=False)
)

salida = salida.str.replace(
    r"\s+a\.\s*m\.$", " AM", regex=True
).str.replace(
    r"\s+p\.\s*m\.$", " PM", regex=True
)

# El signo ? se convierte en dato faltante
salida = salida.mask(salida == "?")

df_etl["Fecha Radicado salida"] = pd.to_datetime(
    salida,
    format="%d/%m/%Y %I:%M:%S %p",
    errors="coerce"
)

# 4. Verificar las conversiones
for columna in [
    "Fecha radicación",
    "Fecha vencimiento",
    "Fecha Radicado salida"
]:
    print(f"\n{columna}")
    print("Tipo de dato:", df_etl[columna].dtype)
    print("Valores faltantes:", df_etl[columna].isna().sum())
    print("Fechas válidas:", df_etl[columna].notna().sum())

display(df_etl[
    ["Fecha radicación", "Fecha vencimiento",
     "Fecha Radicado salida"]
].head(10))


In [ ]:

# Variables categóricas y de identificación almacenadas como texto
columnas_texto = df_etl.select_dtypes(
    include=["object", "string"]
).columns

# Revisar espacios al inicio y al final
print("Espacios al inicio o al final:")
for columna in columnas_texto:
    serie = df_etl[columna].dropna().astype(str)
    cantidad = (
        serie != serie.str.strip()
    ).sum()

    if cantidad > 0:
        print(f"{columna}: {cantidad}")

# Revisar categorías más frecuentes
columnas_categoricas = [
    "Tipo comunicación",
    "Clase",
    "Tipo documento",
    "Canal de recepción",
    "Distribución",
    "Dependencia asignada",
    "Departamento",
    "Municipio"
]

for columna in columnas_categoricas:
    print(f"\n--- {columna} ---")
    display(
        df_etl[columna]
        .value_counts(dropna=False)
        .head(15)
        .rename_axis("categoría")
        .reset_index(name="frecuencia")
    )


In [ ]:

# Crear una copia antes de aplicar nuevas transformaciones
df_etl = df_etl.copy()

# Registrar el estado previo
filas_antes = len(df_etl)
columnas_texto = df_etl.select_dtypes(
    include=["object", "string"]
).columns.tolist()

# Contar espacios sobrantes antes de la limpieza
reporte_textos = []

for columna in columnas_texto:
    serie = df_etl[columna].astype("string")
    con_espacios = serie.notna() & serie.ne(serie.str.strip())

    reporte_textos.append({
        "variable": columna,
        "valores_con_espacios_sobrantes": int(con_espacios.sum())
    })

reporte_textos = pd.DataFrame(reporte_textos)

print("Espacios sobrantes identificados antes de limpiar:")
display(reporte_textos)

# Limpiar únicamente espacios al inicio y al final
for columna in columnas_texto:
    df_etl[columna] = df_etl[columna].map(
        lambda valor: valor.strip()
        if isinstance(valor, str) else valor
    )

# Verificar que no se hayan eliminado registros
print("Filas antes:", filas_antes)
print("Filas después:", len(df_etl))
print("Columnas:", df_etl.shape[1])
print("Duplicados exactos:", df_etl.duplicated().sum())


In [ ]:

# Verificar duplicados en el número de radicado
col_radicado = "No. de Radicado"

print(
    "Radicados vacíos:",
    df_etl[col_radicado].isna().sum()
)

print(
    "Registros con radicado repetido:",
    df_etl[col_radicado].duplicated(keep=False).sum()
)

# Revisar fechas de salida anteriores a la radicación
salida_anterior = (
    df_etl["Fecha Radicado salida"].notna()
    & df_etl["Fecha radicación"].notna()
    & (
        df_etl["Fecha Radicado salida"]
        < df_etl["Fecha radicación"]
    )
)

print(
    "\nRegistros cuya fecha de salida es anterior a la radicación:",
    int(salida_anterior.sum())
)

# Revisar vencimientos anteriores a la radicación
vencimiento_anterior = (
    df_etl["Fecha vencimiento"].notna()
    & df_etl["Fecha radicación"].notna()
    & (
        df_etl["Fecha vencimiento"]
        < df_etl["Fecha radicación"]
    )
)

print(
    "Registros cuyo vencimiento es anterior a la radicación:",
    int(vencimiento_anterior.sum())
)


In [ ]:

# Identificar registros con fecha de salida anterior a la radicación
casos_fecha_inconsistente = df_etl.loc[
    df_etl["Fecha Radicado salida"].notna()
    & df_etl["Fecha radicación"].notna()
    & (
        df_etl["Fecha Radicado salida"]
        < df_etl["Fecha radicación"]
    ),
    [
        "No. de Radicado",
        "Fecha radicación",
        "No. Radicado de salida",
        "Fecha Radicado salida",
        "Fecha vencimiento",
        "Dependencia asignada"
    ]
].copy()

print("Cantidad de casos:", len(casos_fecha_inconsistente))
display(casos_fecha_inconsistente.head(15))


In [ ]:

# Guardar la base auditada en formato CSV
nombre_salida = "base_datos_auditada_etl.csv"

df_etl.to_csv(
    nombre_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:", nombre_salida)
print("Dimensiones:", df_etl.shape)

# Descargar el archivo a tu computador
from google.colab import files
files.download(nombre_salida)


In [ ]:

casos_fecha_inconsistente = df_etl.loc[
    df_etl["Fecha Radicado salida"].notna()
    & df_etl["Fecha radicación"].notna()
    & (
        df_etl["Fecha Radicado salida"]
        < df_etl["Fecha radicación"]
    ),
    [
        "No. de Radicado",
        "Fecha radicación",
        "No. Radicado de salida",
        "Fecha Radicado salida",
        "Fecha vencimiento",
        "Dependencia asignada"
    ]
].copy()

print("Cantidad de casos:", len(casos_fecha_inconsistente))

display(casos_fecha_inconsistente.head(15))


In [ ]:

# Registrar las inconsistencias temporales sin modificar los datos
df_etl["Alerta_fecha_salida"] = (
    df_etl["Fecha Radicado salida"].notna()
    & df_etl["Fecha radicación"].notna()
    & (
        df_etl["Fecha Radicado salida"]
        < df_etl["Fecha radicación"]
    )
)

# Resumen de la auditoría
resumen_alertas = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Fechas de salida anteriores a la radicación",
        "Fechas de salida iguales o posteriores a la radicación",
        "Registros sin fecha de salida"
    ],
    "Cantidad": [
        len(df_etl),
        int(df_etl["Alerta_fecha_salida"].sum()),
        int((
            df_etl["Fecha Radicado salida"].notna()
            & ~df_etl["Alerta_fecha_salida"]
        ).sum()),
        int(df_etl["Fecha Radicado salida"].isna().sum())
    ]
})

display(resumen_alertas)


In [ ]:

# Tabla consolidada de calidad de los datos

auditoria_final = pd.DataFrame({
    "Indicador": [
        "Número de registros",
        "Número de variables originales",
        "Duplicados exactos",
        "Radicados vacíos",
        "Radicados repetidos",
        "Nombres con espacios sobrantes identificados",
        "Faltantes en nombre del peticionario",
        "Faltantes en distribución",
        "Faltantes en fecha de vencimiento",
        "Faltantes en número de radicado de salida",
        "Fechas de salida anteriores a la radicación"
    ],
    "Resultado": [
        len(df_etl),
        15,
        int(df_etl.drop(columns=["Alerta_fecha_salida"]).duplicated().sum()),
        int(df_etl["No. de Radicado"].isna().sum()),
        int(df_etl["No. de Radicado"].duplicated().sum()),
        1557,
        int(df_etl["Nombre del Peticionario / Remitente"].isna().sum()),
        int(df_etl["Distribución"].isna().sum()),
        int(df_etl["Fecha vencimiento"].isna().sum()),
        int(df_etl["No. Radicado de salida"].isna().sum()),
        int(df_etl["Alerta_fecha_salida"].sum())
    ]
})

display(auditoria_final)


In [ ]:

# Exportar la base procesada
archivo_etl = "base_datos_auditada_etl.csv"

df_etl.to_csv(
    archivo_etl,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo exportado:", archivo_etl)
print("Registros:", df_etl.shape[0])
print("Variables:", df_etl.shape[1])

from google.colab import files
files.download(archivo_etl)


In [ ]:

registro_etl = pd.DataFrame([
    {
        "Etapa": "Carga",
        "Operacion": "Lectura del archivo CSV",
        "Resultado": "33.842 registros y 15 variables originales"
    },
    {
        "Etapa": "Auditoría",
        "Operacion": "Revisión de duplicados exactos",
        "Resultado": "No se encontraron duplicados exactos"
    },
    {
        "Etapa": "Auditoría",
        "Operacion": "Revisión de radicados",
        "Resultado": "Sin radicados vacíos ni repetidos"
    },
    {
        "Etapa": "Limpieza",
        "Operacion": "Eliminación de espacios al inicio y al final",
        "Resultado": "Se identificaron 1.557 nombres con espacios sobrantes"
    },
    {
        "Etapa": "Transformación",
        "Operacion": "Conversión de las tres columnas de fecha",
        "Resultado": "Fechas convertidas a datetime; se conservaron los faltantes"
    },
    {
        "Etapa": "Auditoría temporal",
        "Operacion": "Identificación de fechas de salida anteriores a la radicación",
        "Resultado": "97 registros marcados con una alerta, sin modificar las fechas"
    }
])

display(registro_etl)

registro_etl.to_csv(
    "registro_transformaciones_etl.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Registro ETL guardado correctamente.")


In [ ]:
import pandas as pd
import os

print(os.listdir("/content"))


In [ ]:
import pandas as pd

base_verificacion = pd.read_csv(
    "/content/base_datos_auditada_etl.csv"
)

print("Número de registros:", base_verificacion.shape[0])
print("Número de columnas:", base_verificacion.shape[1])

print("\nColumnas de la base:")
print(base_verificacion.columns.tolist())

print("\nPrimeras cinco filas:")
display(base_verificacion.head())

In [ ]:
import pandas as pd

base_verificacion = pd.read_csv(
    "/content/base_datos_auditada_etl.csv"
)

print("DIMENSIONES DE LA BASE")
print("Registros:", len(base_verificacion))
print("Columnas:", len(base_verificacion.columns))

print("\nVALORES FALTANTES POR VARIABLE")
faltantes = base_verificacion.isna().sum()
display(faltantes[faltantes > 0].to_frame("Valores faltantes"))

print("\nREGISTROS DUPLICADOS")
print(
    "Duplicados exactos:",
    base_verificacion.duplicated(
        subset=base_verificacion.columns.drop("Alerta_fecha_salida")
    ).sum()
)

print("\nALERTAS DE FECHA")
print(
    base_verificacion["Alerta_fecha_salida"]
    .value_counts(dropna=False)
)

print("\nTIPOS DE DATOS")
display(
    base_verificacion.dtypes.to_frame("Tipo de dato")
)

In [ ]:
import pandas as pd

base_verificacion = pd.read_csv(
    "/content/base_datos_auditada_etl.csv"
)

# Convertir las fechas al tipo datetime
columnas_fecha = [
    "Fecha radicación",
    "Fecha vencimiento",
    "Fecha Radicado salida"
]

for columna in columnas_fecha:
    base_verificacion[columna] = pd.to_datetime(
        base_verificacion[columna],
        errors="coerce"
    )

# Verificar los cambios
print("TIPOS DE DATOS DESPUÉS DE LA CONVERSIÓN")
display(base_verificacion.dtypes.to_frame("Tipo de dato"))

print("\nVALORES FALTANTES EN LAS FECHAS")
display(
    base_verificacion[columnas_fecha]
    .isna()
    .sum()
    .to_frame("Valores faltantes")
)

print("\nREGISTROS CON ALERTA DE FECHA")
print(
    base_verificacion["Alerta_fecha_salida"].value_counts()
)

In [ ]:
# Guardar la base con las fechas normalizadas
base_verificacion.to_csv(
    "/content/base_datos_auditada_etl_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Base final guardada correctamente.")
print("Registros:", base_verificacion.shape[0])
print("Columnas:", base_verificacion.shape[1])

In [ ]:
import os

print(
    "Archivo encontrado:",
    os.path.exists("/content/base_datos_auditada_etl_final.csv")
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar la base final
df = pd.read_csv(
    "/content/base_datos_auditada_etl_final.csv",
    parse_dates=[
        "Fecha radicación",
        "Fecha vencimiento",
        "Fecha Radicado salida"
    ]
)

print("Dimensiones de la base:", df.shape)

# 1. Distribución por clase
print("\nRADICADOS POR CLASE")
tabla_clase = df["Clase"].value_counts(dropna=False).to_frame("Cantidad")
tabla_clase["Porcentaje"] = (
    tabla_clase["Cantidad"] / len(df) * 100
).round(2)
display(tabla_clase)

# 2. Distribución por canal
print("\nRADICADOS POR CANAL DE RECEPCIÓN")
tabla_canal = df["Canal de recepción"].value_counts(dropna=False).to_frame("Cantidad")
tabla_canal["Porcentaje"] = (
    tabla_canal["Cantidad"] / len(df) * 100
).round(2)
display(tabla_canal)

# 3. Distribución mensual
df["Mes_radicacion"] = df["Fecha radicación"].dt.to_period("M").astype(str)
tabla_mensual = df["Mes_radicacion"].value_counts().sort_index().to_frame("Cantidad")
display(tabla_mensual)

In [ ]:
# Gráfica 1: comunicaciones por clase
tabla_clase["Cantidad"].plot(kind="bar", rot=0)
plt.title("Distribución de comunicaciones por clase")
plt.xlabel("Clase")
plt.ylabel("Número de radicados")
plt.tight_layout()
plt.show()

# Gráfica 2: comunicaciones por canal
tabla_canal["Cantidad"].plot(kind="bar", rot=30)
plt.title("Distribución de comunicaciones por canal de recepción")
plt.xlabel("Canal de recepción")
plt.ylabel("Número de radicados")
plt.tight_layout()
plt.show()

# Gráfica 3: radicados por mes
tabla_mensual["Cantidad"].plot(kind="line", marker="o")
plt.title("Evolución mensual de las radicaciones")
plt.xlabel("Mes")
plt.ylabel("Número de radicados")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Diez dependencias con mayor cantidad de radicados
print("TOP 10 DEPENDENCIAS POR NÚMERO DE RADICADOS")

tabla_dependencias = (
    df["Dependencia asignada"]
    .value_counts(dropna=False)
    .head(10)
    .to_frame("Cantidad")
)

tabla_dependencias["Porcentaje del total"] = (
    tabla_dependencias["Cantidad"] / len(df) * 100
).round(2)

display(tabla_dependencias)


# 2. Presencia o ausencia de número de radicado de salida
print("\nREGISTROS SEGÚN DISPONIBILIDAD DEL RADICADO DE SALIDA")

df["Estado_registro_salida"] = df["No. Radicado de salida"].notna().map({
    True: "Con radicado de salida registrado",
    False: "Sin radicado de salida registrado"
})

tabla_salida = df["Estado_registro_salida"].value_counts().to_frame("Cantidad")

tabla_salida["Porcentaje"] = (
    tabla_salida["Cantidad"] / len(df) * 100
).round(2)

display(tabla_salida)

In [ ]:
# Tabla cruzada entre clase y disponibilidad del radicado de salida
tabla_cruzada = pd.crosstab(
    df["Clase"],
    df["Estado_registro_salida"],
    margins=True
)

print("CANTIDAD DE REGISTROS")
display(tabla_cruzada)

# Porcentaje dentro de cada clase
tabla_porcentajes = pd.crosstab(
    df["Clase"],
    df["Estado_registro_salida"],
    normalize="index"
).mul(100).round(2)

print("\nPORCENTAJE DENTRO DE CADA CLASE")
display(tabla_porcentajes)

In [ ]:
# Crear una copia para el análisis temporal
df_temporal = df.copy()

# Conservar registros con ambas fechas disponibles
df_temporal = df_temporal[
    df_temporal["Fecha radicación"].notna()
    & df_temporal["Fecha Radicado salida"].notna()
].copy()

# Calcular diferencia en días calendario
df_temporal["Dias_entre_radicacion_y_salida"] = (
    df_temporal["Fecha Radicado salida"]
    - df_temporal["Fecha radicación"]
).dt.total_seconds() / 86400

print("Registros con ambas fechas:", len(df_temporal))

print("\nResumen de días entre radicación y salida")
display(
    df_temporal["Dias_entre_radicacion_y_salida"]
    .describe()
    .to_frame("Días")
)

print("\nRegistros con salida anterior a la radicación:")
print(
    (df_temporal["Dias_entre_radicacion_y_salida"] < 0).sum()
)

In [ ]:
# Separar registros para el análisis, sin modificar la base original
df_tiempos_validos = df_temporal[
    df_temporal["Dias_entre_radicacion_y_salida"] >= 0
].copy()

print("RESUMEN DE TIEMPOS NO NEGATIVOS")
print("Registros analizados:", len(df_tiempos_validos))

display(
    df_tiempos_validos["Dias_entre_radicacion_y_salida"]
    .describe()
    .to_frame("Días")
)

print("\nREGISTROS CON DIFERENCIAS NEGATIVAS PARA REVISIÓN")

df_fechas_inconsistentes = df_temporal[
    df_temporal["Dias_entre_radicacion_y_salida"] < 0
].copy()

print("Cantidad:", len(df_fechas_inconsistentes))

display(
    df_fechas_inconsistentes[
        [
            "No. de Radicado",
            "Fecha radicación",
            "Fecha Radicado salida",
            "Dias_entre_radicacion_y_salida"
        ]
    ].head(10)
)

In [ ]:
# Seleccionar registros que tienen fecha de salida y fecha de vencimiento
df_vencimientos = df[
    df["Fecha Radicado salida"].notna()
    & df["Fecha vencimiento"].notna()
].copy()

# Calcular diferencia en días entre salida y vencimiento
df_vencimientos["Dias_salida_vs_vencimiento"] = (
    df_vencimientos["Fecha Radicado salida"]
    - df_vencimientos["Fecha vencimiento"]
).dt.total_seconds() / 86400

# Clasificar los registros
df_vencimientos["Clasificacion_fecha"] = "Salida en fecha de vencimiento"

df_vencimientos.loc[
    df_vencimientos["Dias_salida_vs_vencimiento"] < 0,
    "Clasificacion_fecha"
] = "Salida anterior al vencimiento"

df_vencimientos.loc[
    df_vencimientos["Dias_salida_vs_vencimiento"] > 0,
    "Clasificacion_fecha"
] = "Salida posterior al vencimiento"

# Mostrar cantidades y porcentajes
tabla_vencimientos = (
    df_vencimientos["Clasificacion_fecha"]
    .value_counts()
    .to_frame("Cantidad")
)

tabla_vencimientos["Porcentaje"] = (
    tabla_vencimientos["Cantidad"]
    / len(df_vencimientos) * 100
).round(2)

print("Registros con fecha de salida y vencimiento:",
      len(df_vencimientos))

display(tabla_vencimientos)

print("\nResumen de días entre salida y vencimiento:")
display(
    df_vencimientos["Dias_salida_vs_vencimiento"]
    .describe()
    .to_frame("Días")
)

In [ ]:
# Excluir del análisis los registros con salida anterior a la radicación
df_vencimientos_validos = df_vencimientos[
    df_vencimientos["Dias_entre_radicacion_y_salida"] >= 0
].copy()

# Clasificar según la fecha de salida frente al vencimiento
df_vencimientos_validos["Estado_vencimiento"] = "Salida posterior al vencimiento"

df_vencimientos_validos.loc[
    df_vencimientos_validos["Dias_salida_vs_vencimiento"] < 0,
    "Estado_vencimiento"
] = "Salida anterior al vencimiento"

df_vencimientos_validos.loc[
    df_vencimientos_validos["Dias_salida_vs_vencimiento"] == 0,
    "Estado_vencimiento"
] = "Salida en fecha de vencimiento"

# Tabla de cantidades por clase
tabla_clase_vencimiento = pd.crosstab(
    df_vencimientos_validos["Clase"],
    df_vencimientos_validos["Estado_vencimiento"],
    margins=True
)

print("CANTIDADES POR CLASE Y ESTADO DE VENCIMIENTO")
display(tabla_clase_vencimiento)

# Porcentajes dentro de cada clase
tabla_clase_vencimiento_pct = pd.crosstab(
    df_vencimientos_validos["Clase"],
    df_vencimientos_validos["Estado_vencimiento"],
    normalize="index"
).mul(100).round(2)

print("\nPORCENTAJES DENTRO DE CADA CLASE")
display(tabla_clase_vencimiento_pct)

print("\nRegistros incluidos:", len(df_vencimientos_validos))
print("Registros excluidos solo de este análisis:",
      len(df_vencimientos) - len(df_vencimientos_validos))

In [ ]:
import pandas as pd

# Seleccionar registros con fecha de salida y vencimiento
df_vencimientos = df[
    df["Fecha Radicado salida"].notna()
    & df["Fecha vencimiento"].notna()
].copy()

# Calcular nuevamente la diferencia entre salida y radicación
df_vencimientos["Dias_entre_radicacion_y_salida"] = (
    df_vencimientos["Fecha Radicado salida"]
    - df_vencimientos["Fecha radicación"]
).dt.total_seconds() / 86400

# Calcular diferencia entre salida y vencimiento
df_vencimientos["Dias_salida_vs_vencimiento"] = (
    df_vencimientos["Fecha Radicado salida"]
    - df_vencimientos["Fecha vencimiento"]
).dt.total_seconds() / 86400

# Excluir del análisis temporal las salidas anteriores a la radicación
df_vencimientos_validos = df_vencimientos[
    df_vencimientos["Dias_entre_radicacion_y_salida"] >= 0
].copy()

# Clasificar según la fecha de salida frente al vencimiento
df_vencimientos_validos["Estado_vencimiento"] = "Salida posterior al vencimiento"

df_vencimientos_validos.loc[
    df_vencimientos_validos["Dias_salida_vs_vencimiento"] < 0,
    "Estado_vencimiento"
] = "Salida anterior al vencimiento"

df_vencimientos_validos.loc[
    df_vencimientos_validos["Dias_salida_vs_vencimiento"] == 0,
    "Estado_vencimiento"
] = "Salida en fecha de vencimiento"

# Mostrar cantidades por clase
tabla_clase_vencimiento = pd.crosstab(
    df_vencimientos_validos["Clase"],
    df_vencimientos_validos["Estado_vencimiento"],
    margins=True
)

print("CANTIDADES POR CLASE Y ESTADO DE VENCIMIENTO")
display(tabla_clase_vencimiento)

# Mostrar porcentajes por clase
tabla_clase_vencimiento_pct = pd.crosstab(
    df_vencimientos_validos["Clase"],
    df_vencimientos_validos["Estado_vencimiento"],
    normalize="index"
).mul(100).round(2)

print("\nPORCENTAJES DENTRO DE CADA CLASE")
display(tabla_clase_vencimiento_pct)

print("\nRegistros incluidos:", len(df_vencimientos_validos))
print(
    "Registros excluidos solo de este análisis:",
    len(df_vencimientos) - len(df_vencimientos_validos)
)

In [ ]:
tabla_dependencia_vencimiento = pd.crosstab(
    df_vencimientos_validos["Dependencia asignada"],
    df_vencimientos_validos["Estado_vencimiento"]
)

# Calcular el total de registros por dependencia
tabla_dependencia_vencimiento["Total"] = (
    tabla_dependencia_vencimiento.sum(axis=1)
)

# Calcular el porcentaje de salidas posteriores al vencimiento
tabla_dependencia_vencimiento["Porcentaje posterior"] = (
    tabla_dependencia_vencimiento["Salida posterior al vencimiento"]
    / tabla_dependencia_vencimiento["Total"] * 100
).round(2)

# Ordenar por cantidad de salidas posteriores
tabla_dependencia_vencimiento = (
    tabla_dependencia_vencimiento
    .sort_values(
        "Salida posterior al vencimiento",
        ascending=False
    )
)

display(tabla_dependencia_vencimiento.head(10))

In [ ]:
tabla_canal_vencimiento = pd.crosstab(
    df_vencimientos_validos["Canal de recepción"],
    df_vencimientos_validos["Estado_vencimiento"]
)

tabla_canal_vencimiento["Total"] = (
    tabla_canal_vencimiento.sum(axis=1)
)

tabla_canal_vencimiento["Porcentaje posterior"] = (
    tabla_canal_vencimiento["Salida posterior al vencimiento"]
    / tabla_canal_vencimiento["Total"] * 100
).round(2)

tabla_canal_vencimiento = tabla_canal_vencimiento.sort_values(
    "Porcentaje posterior",
    ascending=False
)

display(tabla_canal_vencimiento)

In [ ]:

import pandas as pd

# Crear una copia para el análisis temporal
df_tiempos = df.copy()

# Asegurar que las fechas tengan formato datetime
columnas_fecha = [
    "Fecha radicación",
    "Fecha Radicado salida",
    "Fecha vencimiento"
]

for columna in columnas_fecha:
    df_tiempos[columna] = pd.to_datetime(
        df_tiempos[columna],
        errors="coerce"
    )

# Conservar registros con fechas de radicación y salida disponibles
df_tiempos = df_tiempos[
    df_tiempos["Fecha radicación"].notna()
    & df_tiempos["Fecha Radicado salida"].notna()
].copy()

# Calcular el tiempo transcurrido en días
df_tiempos["Dias_entre_radicacion_y_salida"] = (
    df_tiempos["Fecha Radicado salida"]
    - df_tiempos["Fecha radicación"]
).dt.total_seconds() / 86400

# Separar los registros cronológicamente válidos
df_tiempos_validos = df_tiempos[
    df_tiempos["Dias_entre_radicacion_y_salida"] >= 0
].copy()

# Resumen estadístico de los tiempos
resumen_tiempos = (
    df_tiempos_validos["Dias_entre_radicacion_y_salida"]
    .describe()
    .round(2)
)

display(resumen_tiempos)

print("Registros con ambas fechas:", len(df_tiempos))
print("Registros con fechas cronológicamente válidas:",
      len(df_tiempos_validos))
print("Registros excluidos por fecha de salida anterior:",
      len(df_tiempos) - len(df_tiempos_validos))


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(
    df_tiempos_validos["Dias_entre_radicacion_y_salida"],
    bins=30,
    edgecolor="black"
)

plt.title("Distribución del tiempo entre radicación y salida")
plt.xlabel("Días transcurridos")
plt.ylabel("Cantidad de registros")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

# Obtener cuartiles y calcular el rango intercuartílico
Q1 = df_tiempos_validos[
    "Dias_entre_radicacion_y_salida"
].quantile(0.25)

Q3 = df_tiempos_validos[
    "Dias_entre_radicacion_y_salida"
].quantile(0.75)

IQR = Q3 - Q1

# Límites estadísticos para identificar posibles valores atípicos
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Identificar los registros que superan los límites
posibles_atipicos = df_tiempos_validos[
    (df_tiempos_validos["Dias_entre_radicacion_y_salida"] < limite_inferior)
    | (df_tiempos_validos["Dias_entre_radicacion_y_salida"] > limite_superior)
].copy()

print(f"Q1: {Q1:.2f} días")
print(f"Q3: {Q3:.2f} días")
print(f"Rango intercuartílico: {IQR:.2f} días")
print(f"Límite superior: {limite_superior:.2f} días")
print(f"Posibles valores atípicos: {len(posibles_atipicos)}")
print(
    f"Porcentaje de posibles atípicos: "
    f"{len(posibles_atipicos) / len(df_tiempos_validos) * 100:.2f}%"
)

# Diagrama de caja
plt.figure(figsize=(10, 4))
plt.boxplot(
    df_tiempos_validos["Dias_entre_radicacion_y_salida"],
    vert=False
)
plt.title("Diagrama de caja de los tiempos entre radicación y salida")
plt.xlabel("Días transcurridos")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Resumen estadístico de los tiempos por clase
resumen_tiempos_clase = (
    df_tiempos_validos
    .groupby("Clase")["Dias_entre_radicacion_y_salida"]
    .agg(
        Registros="count",
        Promedio="mean",
        Mediana="median",
        Desviacion_estandar="std",
        Minimo="min",
        Maximo="max"
    )
    .round(2)
)

display(resumen_tiempos_clase)


In [ ]:

import matplotlib.pyplot as plt

datos_otro = df_tiempos_validos.loc[
    df_tiempos_validos["Clase"] == "OTRO",
    "Dias_entre_radicacion_y_salida"
].dropna()

datos_pqrsd = df_tiempos_validos.loc[
    df_tiempos_validos["Clase"] == "PQRSD",
    "Dias_entre_radicacion_y_salida"
].dropna()

plt.figure(figsize=(9, 5))

plt.boxplot(
    [datos_otro, datos_pqrsd],
    tick_labels=["OTRO", "PQRSD"],
    showfliers=False
)

plt.title("Comparación de tiempos entre radicación y salida por clase")
plt.xlabel("Clase de comunicación")
plt.ylabel("Días transcurridos")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# Crear una copia para el análisis mensual
df_mensual = df_tiempos_validos.copy()

# Extraer el mes de la fecha de radicación
df_mensual["Mes"] = (
    df_mensual["Fecha radicación"].dt.month
)

# Calcular cantidad de registros y tiempos por mes
resumen_mensual = (
    df_mensual
    .groupby("Mes")
    .agg(
        Registros=("Dias_entre_radicacion_y_salida", "count"),
        Promedio_dias=("Dias_entre_radicacion_y_salida", "mean"),
        Mediana_dias=("Dias_entre_radicacion_y_salida", "median")
    )
    .round(2)
)

# Mostrar los meses en orden cronológico
nombres_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo",
    4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre",
    10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

resumen_mensual.index = [
    nombres_meses[mes] for mes in resumen_mensual.index
]

display(resumen_mensual)


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))

plt.plot(
    resumen_mensual.index,
    resumen_mensual["Promedio_dias"],
    marker="o",
    label="Promedio"
)

plt.plot(
    resumen_mensual.index,
    resumen_mensual["Mediana_dias"],
    marker="s",
    label="Mediana"
)

plt.title("Evolución mensual de los tiempos entre radicación y salida")
plt.xlabel("Mes de radicación")
plt.ylabel("Días transcurridos")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Resumir los tiempos por mes y clase
df_mensual_clase = df_tiempos_validos.copy()

df_mensual_clase["Mes_numero"] = (
    df_mensual_clase["Fecha radicación"].dt.month
)

resumen_mensual_clase = (
    df_mensual_clase
    .groupby(["Mes_numero", "Clase"])
    .agg(
        Registros=("Dias_entre_radicacion_y_salida", "count"),
        Promedio_dias=("Dias_entre_radicacion_y_salida", "mean"),
        Mediana_dias=("Dias_entre_radicacion_y_salida", "median")
    )
    .round(2)
    .reset_index()
)

nombres_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo",
    4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre",
    10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

resumen_mensual_clase["Mes"] = (
    resumen_mensual_clase["Mes_numero"].map(nombres_meses)
)

display(
    resumen_mensual_clase[
        ["Mes", "Clase", "Registros", "Promedio_dias", "Mediana_dias"]
    ]
)


In [ ]:

import matplotlib.pyplot as plt

# Preparar los datos para graficar
tabla_grafica = resumen_mensual_clase.pivot(
    index="Mes_numero",
    columns="Clase",
    values="Promedio_dias"
).sort_index()

nombres_meses_cortos = [
    "Ene", "Feb", "Mar", "Abr", "May", "Jun",
    "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"
]

plt.figure(figsize=(11, 5))

for clase in tabla_grafica.columns:
    plt.plot(
        range(1, 13),
        tabla_grafica[clase],
        marker="o",
        label=clase
    )

plt.title("Promedio mensual de días entre radicación y salida por clase")
plt.xlabel("Mes de radicación")
plt.ylabel("Promedio de días")
plt.xticks(range(1, 13), nombres_meses_cortos)
plt.legend(title="Clase")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Comparar dependencias con un volumen mínimo de 100 registros
resumen_dependencias = (
    df_vencimientos_validos
    .groupby("Dependencia asignada")
    .agg(
        Total=("Estado_vencimiento", "count"),
        Salidas_posteriores=(
            "Estado_vencimiento",
            lambda x: (x == "Salida posterior al vencimiento").sum()
        )
    )
)

resumen_dependencias["Porcentaje_posterior"] = (
    resumen_dependencias["Salidas_posteriores"]
    / resumen_dependencias["Total"] * 100
).round(2)

resumen_dependencias = resumen_dependencias[
    resumen_dependencias["Total"] >= 100
].sort_values(
    "Porcentaje_posterior",
    ascending=False
)

display(resumen_dependencias.head(10))


In [ ]:

import pandas as pd
from pathlib import Path

# Cargar la base final del proceso ETL
ruta = Path("/content/base_datos_auditada_etl_final.csv")

if not ruta.exists():
    print("No se encontró el archivo:", ruta)
else:
    base_final = pd.read_csv(
        ruta,
        parse_dates=[
            "Fecha radicación",
            "Fecha vencimiento",
            "Fecha Radicado salida"
        ]
    )

    print("DIMENSIONES DE LA BASE FINAL")
    print("Registros:", base_final.shape[0])
    print("Variables:", base_final.shape[1])

    print("\nTIPOS DE DATOS")
    display(base_final.dtypes.to_frame("Tipo de dato"))

    print("\nVALORES FALTANTES")
    faltantes = base_final.isna().sum().to_frame("Cantidad faltante")
    faltantes["Porcentaje (%)"] = (
        faltantes["Cantidad faltante"] / len(base_final) * 100
    ).round(2)

    display(faltantes[faltantes["Cantidad faltante"] > 0])

    print("\nVERIFICACIÓN DE ALERTAS TEMPORALES")
    if "Alerta_fecha_salida" in base_final.columns:
        print(
            "Registros con fecha de salida anterior a la radicación:",
            int(base_final["Alerta_fecha_salida"].sum())
        )
    else:
        print("La variable Alerta_fecha_salida no está en el archivo.")


In [ ]:

# Verificación de integridad de la base final

print("REGISTROS TOTALES:", len(base_final))

print(
    "Filas duplicadas completas:",
    base_final.duplicated().sum()
)

print(
    "Radicados de entrada faltantes:",
    base_final["No. de Radicado"].isna().sum()
)

print(
    "Radicados de entrada duplicados:",
    base_final["No. de Radicado"].duplicated().sum()
)

print(
    "Radicados de entrada únicos:",
    base_final["No. de Radicado"].nunique()
)


In [ ]:

# Resumen de las transformaciones realizadas

print("BASE ORIGINAL")
print("Registros:", 33842)
print("Variables originales:", 15)

print("\nBASE FINAL")
print("Registros:", base_final.shape[0])
print("Variables finales:", base_final.shape[1])

print("\nVARIABLES INCORPORADAS")
variables_nuevas = [
    col for col in base_final.columns
    if col not in [
        "No. de Radicado",
        "Tipo comunicación",
        "Clase",
        "Tipo documento",
        "Canal de recepción",
        "Nombre del Peticionario / Remitente",
        "Distribución",
        "Dependencia asignada",
        "Departamento",
        "Municipio",
        "No. Radicado de salida",
        "Fecha radicación",
        "Identificacion",
        "Fecha vencimiento",
        "Fecha Radicado salida"
    ]
]
print(variables_nuevas)

print("\nVARIABLES DE FECHA")
for columna in [
    "Fecha radicación",
    "Fecha vencimiento",
    "Fecha Radicado salida"
]:
    print(f"{columna}: {base_final[columna].dtype}")

print("\nALERTAS TEMPORALES")
print(base_final["Alerta_fecha_salida"].value_counts(dropna=False))


In [ ]:
# Diccionario de datos: estructura y valores faltantes

diccionario = pd.DataFrame({
    "Variable": base_final.columns,
    "Tipo de dato": base_final.dtypes.astype(str).values,
    "Valores faltantes": base_final.isna().sum().values,
    "Porcentaje faltante (%)": (
        base_final.isna().mean().values * 100
    ).round(2),
    "Valores únicos": base_final.nunique().values
})

display(diccionario)

In [ ]:
import pandas as pd

In [ ]:
diccionario = pd.DataFrame({
    "Variable": base_final.columns,
    "Tipo de dato": base_final.dtypes.astype(str).values,
    "Valores faltantes": base_final.isna().sum().values,
    "Porcentaje faltante (%)": (
        base_final.isna().mean().values * 100
    ).round(2),
    "Valores únicos": base_final.nunique().values
})

display(diccionario)

In [ ]:
import pandas as pd

base_final = pd.read_csv(
    "/content/base_datos_auditada_etl_final.csv",
    parse_dates=[
        "Fecha radicación",
        "Fecha vencimiento",
        "Fecha Radicado salida"
    ]
)

print("Base cargada correctamente")
print("Dimensiones:", base_final.shape)

In [ ]:
import os

print("Archivos disponibles en /content:")
for archivo in os.listdir("/content"):
    print(archivo)

In [ ]:
from google.colab import files

archivos = files.upload()

In [ ]:
import pandas as pd

base_final = pd.read_csv(
    "/content/base_datos_auditada_etl_final.csv",
    parse_dates=[
        "Fecha radicación",
        "Fecha vencimiento",
        "Fecha Radicado salida"
    ]
)

print("Dimensiones:", base_final.shape)
display(base_final.head())

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import pandas as pd

base_final = pd.read_csv(
    "/content/base_datos_auditada_etl (1).csv",
    parse_dates=[
        "Fecha radicación",
        "Fecha vencimiento",
        "Fecha Radicado salida"
    ]
)

print("Base cargada correctamente")
print("Dimensiones:", base_final.shape)

In [ ]:
diccionario = pd.DataFrame({
    "Variable": base_final.columns,
    "Tipo de dato": base_final.dtypes.astype(str).values,
    "Valores faltantes": base_final.isna().sum().values,
    "Porcentaje faltante (%)": (
        base_final.isna().mean().values * 100
    ).round(2),
    "Valores únicos": base_final.nunique().values
})

display(diccionario)

In [ ]:
print("Valores únicos de Identificacion:")
print(base_final["Identificacion"].value_counts(dropna=False))

print("\nPrimeros registros:")
display(base_final[["Identificacion", "No. de Radicado"]].head(10))

In [ ]:
for columna in base_final.columns:
    cantidad = base_final[columna].nunique()

    if cantidad == 1:
        print(
            f"{columna}: "
            f"{base_final[columna].dropna().unique()[0]}"
        )

In [ ]:
variables_categoricas = [
    "Tipo comunicación",
    "Clase",
    "Tipo documento",
    "Canal de recepción",
    "Distribución",
    "Dependencia asignada",
    "Departamento",
    "Municipio"
]

for variable in variables_categoricas:
    print(f"\n{'='*60}")
    print(f"VARIABLE: {variable}")
    print(f"Valores diferentes: {base_final[variable].nunique()}")
    print(base_final[variable].value_counts(dropna=False).to_string())

In [ ]:
variables_restantes = [
    "No. de Radicado",
    "Nombre del Peticionario / Remitente",
    "No. Radicado de salida",
    "Fecha radicación",
    "Identificacion",
    "Fecha vencimiento",
    "Fecha Radicado salida",
    "Alerta_fecha_salida"
]

for variable in variables_restantes:
    print(f"\n{'='*55}")
    print(f"VARIABLE: {variable}")
    print("Tipo:", base_final[variable].dtype)
    print("Valores únicos:", base_final[variable].nunique())
    print("Valores faltantes:", base_final[variable].isna().sum())
    print("Ejemplos:")
    print(base_final[variable].dropna().head(5).to_list())

In [ ]:
variables_fecha = [
    "Fecha radicación",
    "Fecha vencimiento",
    "Fecha Radicado salida"
]

for variable in variables_fecha:
    print(f"\nVARIABLE: {variable}")
    print("Fecha mínima:", base_final[variable].min())
    print("Fecha máxima:", base_final[variable].max())
    print("Valores faltantes:", base_final[variable].isna().sum())

In [ ]:
# Exportar la base de datos auditada
base_final.to_csv(
    "/content/base_datos_auditada_etl_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Base auditada exportada correctamente.")
print("Dimensiones:", base_final.shape)

In [ ]:
base_final = pd.read_csv(
    "base_datos_auditada_etl (1).csv",
    parse_dates=[
        "Fecha radicación",
        "Fecha vencimiento",
        "Fecha Radicado salida"
    ]
)

print("Base recuperada correctamente")
print("Dimensiones:", base_final.shape)

In [ ]:
base_final.to_csv(
    "/content/base_datos_auditada_etl_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo exportado correctamente.")
print("Filas:", len(base_final))
print("Columnas:", len(base_final.columns))

In [ ]:
from google.colab import files

files.download("/content/base_datos_auditada_etl_final.csv")